# ML Notebook on Monthly Dataset

Ce notebook utilise le dataset mensuel propre construit dans `monthly_entity_panel_ml_ready.csv`.

Variables utilisées :
- `absence_rate`
- `KTI`
- `pct_decay`
- `strategic_share`

Vu qu'il n'y a pas encore de cible supervisée officielle, le notebook applique un clustering `KMeans` pour identifier des profils mensuels d'entités.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


In [ ]:
BASE_DIR = Path(".")
DATA_PATH = BASE_DIR / "outputs" / "monthly_entity_panel_ml_ready.csv"
OUTPUT_XLSX = BASE_DIR / "outputs" / "ml_monthly_model_results.xlsx"

ID_COLS = ["month", "year", "entity", "data_quality_flag"]
FEATURES = ["absence_rate", "KTI", "pct_decay", "strategic_share"]

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)


In [ ]:
df = pd.read_csv(DATA_PATH)
df = df[ID_COLS + FEATURES].copy()
df = df.dropna(subset=FEATURES).reset_index(drop=True)
df["month"] = df["month"].astype(str)

print("Dataset shape:", df.shape)
display(df.head(12))
display(df[FEATURES].describe().T)
display(df.groupby(["entity", "data_quality_flag"]).size().reset_index(name="n_rows"))


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES])

selection_rows = []
for k in range(2, 6):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X_scaled)
    selection_rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette_score": silhouette_score(X_scaled, labels),
    })

selection_df = pd.DataFrame(selection_rows).sort_values("silhouette_score", ascending=False).reset_index(drop=True)
display(selection_df)


In [ ]:
best_k = int(selection_df.iloc[0]["k"])
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
df["cluster"] = kmeans.fit_predict(X_scaled)

print("Best k:", best_k)
display(df.head(20))


In [ ]:
cluster_profile = df.groupby("cluster")[FEATURES].mean().round(4)
cluster_size = df.groupby("cluster").size().reset_index(name="n")
cluster_summary = cluster_size.merge(cluster_profile.reset_index(), on="cluster", how="left").sort_values("cluster")
display(cluster_summary)


In [ ]:
entity_cluster_summary = df.groupby(["entity", "cluster"]).size().reset_index(name="n_rows")
year_cluster_summary = df.groupby(["year", "cluster"]).size().reset_index(name="n_rows")
display(entity_cluster_summary)
display(year_cluster_summary)


In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
pca_df = df[ID_COLS + ["cluster"]].copy()
pca_df["pc1"] = coords[:, 0]
pca_df["pc2"] = coords[:, 1]
print("Explained variance ratio:", pca.explained_variance_ratio_)
display(pca_df.head(20))


In [ ]:
cluster_examples = df.sort_values(["cluster", "absence_rate"], ascending=[True, False]).copy()
display(cluster_examples[["month", "year", "entity", "cluster"] + FEATURES].head(40))


In [ ]:
cluster_interpretation = cluster_summary.copy()
cluster_interpretation["interpretation"] = ""

for idx, row in cluster_interpretation.iterrows():
    parts = []
    parts.append("absence plutot elevee" if row["absence_rate"] >= df["absence_rate"].median() else "absence plutot faible")
    parts.append("KTI plutot eleve" if row["KTI"] >= df["KTI"].median() else "KTI plutot faible")
    parts.append("decay plutot eleve" if row["pct_decay"] >= df["pct_decay"].median() else "decay plutot faible")
    parts.append("formation strategique plutot elevee" if row["strategic_share"] >= df["strategic_share"].median() else "formation strategique plutot faible")
    cluster_interpretation.loc[idx, "interpretation"] = ", ".join(parts)

display(cluster_interpretation)


## Lecture métier

Pour commenter les résultats :
- le `silhouette_score` mesure la netteté de séparation entre les groupes ;
- le `cluster_summary` donne le profil moyen de chaque segment ;
- `entity_cluster_summary` montre quelles entités tombent le plus souvent dans chaque cluster ;
- `year_cluster_summary` montre si les profils changent dans le temps.


In [ ]:
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    selection_df.to_excel(writer, sheet_name="model_selection", index=False)
    df.to_excel(writer, sheet_name="clustered_dataset", index=False)
    cluster_summary.to_excel(writer, sheet_name="cluster_summary", index=False)
    cluster_interpretation.to_excel(writer, sheet_name="cluster_interpretation", index=False)
    entity_cluster_summary.to_excel(writer, sheet_name="entity_cluster_summary", index=False)
    year_cluster_summary.to_excel(writer, sheet_name="year_cluster_summary", index=False)
    pca_df.to_excel(writer, sheet_name="pca_projection", index=False)

print(OUTPUT_XLSX.resolve())
